# Train SARO's PAS policy for Go2 on a single A100

Replicates the low-level locomotion training technique from **SARO** (arXiv:2407.16412) for the Unitree Go2, using [mjlab](https://mujocolab.github.io/mjlab/) + `rsl_rl`. Implementation + architecture docs: `docs/07-pas-implementation.md` in the repo.

Adapted from `kaggle/train_pas.ipynb` for a single-GPU **A100 (40GB)** college server with an **ephemeral filesystem** (nothing survives a job/container restart, same assumption as the Kaggle notebook).

**Before running:**
1. Export `HF_TOKEN` (a Hugging Face **write** token, huggingface.co/settings/tokens) in the shell *before* launching Jupyter -- this notebook reads it from the environment rather than prompting for it. Used to push checkpoints to a model repo and to resume across job restarts.
2. If `github.com/BRUH-MAIN/policyswitching` is private, also export `GITHUB_TOKEN` (a GitHub PAT with `repo` scope) the same way.
3. If your server is shared, activate your own venv/conda env before starting Jupyter -- this notebook installs into whatever kernel it's running under.
4. Edit the CONFIG cell below if you want a different HF repo name, env count, or iteration budget.

**What this notebook does:** clones the repo, installs mjlab/rsl_rl, checks Hugging Face for an existing checkpoint to resume from (so re-running this notebook after the job/container gets killed picks up where it left off), trains Stage 1 (oracle) then Stage 2 (anneal), uploading every checkpoint to your HF model repo as it trains.

In [ ]:
# ==== CONFIG — edit as needed ====
import os

GITHUB_REPO = "BRUH-MAIN/policyswitching"
HF_REPO_NAME = "go2-pas-saro"       # final repo id will be f"{your_hf_username}/{HF_REPO_NAME}"

# Single A100 (40GB): this launcher does NOT split num_envs across GPUs (see kaggle/train_pas.ipynb
# for the multi-GPU case) -- with one GPU there's just one worker, so this is the full VRAM budget.
# 8192 is an untested extrapolation from a confirmed-working 2048 envs on an 8GB local GPU (this
# terrain+heightscan+LSTM-estimator env is heavier than a flat-terrain baseline) -- watch
# `nvidia-smi` during the first few hundred iterations and adjust up/down from there.
NUM_ENVS = 8192                       # reduce (e.g. 4096) if you hit OOM, raise if there's headroom to spare
GPU_IDS = "[0]"                       # single GPU -- runs directly, no torchrunx multi-process launch

# STAGE{1,2}_BUDGET are ABSOLUTE iteration targets, not what gets passed to
# --agent.max-iterations directly -- rsl_rl's `learn(num_learning_iterations=N)` runs N
# MORE iterations from wherever a run resumes (`total_it = start_it + N`), it is not an
# absolute target. Passing the same fixed N on every resume makes the target grow by N
# every restart (this happened for real on the Kaggle run this notebook was adapted from:
# it resumed at iteration 7800 and the next target silently became 7800+40000=47800). The
# stage cells below compute `remaining = budget - iterations_already_done` each time and
# pass THAT instead, so rerunning this notebook after a kill always converges on the same total.
STAGE1_BUDGET = 40000                  # paper default; lower this for a first smoke run, e.g. 2000
STAGE2_BUDGET = 40000                  # counted from stage 1's final iteration, not from 0 (see stage 2 cell)
SAVE_INTERVAL = 200                    # PPO iterations between checkpoints (and HF uploads)

REPO_DIR = os.path.expanduser("~/policyswitching")
MJLAB_DIR = f"{REPO_DIR}/unitree_rl_mjlab"

## 1. Environment check
Fail fast here rather than after a slow install — mjlab needs driver ≥550 and CUDA 12.4+ (MuJoCo Warp is picky about CUDA version).

In [ ]:
!nvidia-smi
!nvcc --version || echo 'nvcc not found (fine if a matching CUDA runtime is still installed via pip)'
!python3 --version

## 2. Clone the repo

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    github_token = os.environ.get("GITHUB_TOKEN")

    if github_token:
        clone_url = f"https://{github_token}@github.com/{GITHUB_REPO}.git"
    else:
        clone_url = f"https://github.com/{GITHUB_REPO}.git"
    !git clone {clone_url} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

!ls {MJLAB_DIR}/src/tasks/velocity/mdp/pas.py && echo 'PAS implementation found.'

## 3. Install dependencies
Installs into whatever Python environment this Jupyter kernel is using.

In [ ]:
%cd {MJLAB_DIR}
!pip install -q -e .
# setup.py pins mjlab==1.2.0 / mujoco-warp==3.5.0 but not an exact mujoco (core) version, so a
# loose resolver can silently pull a newer, incompatible mujoco -- force the matching version
# explicitly (verified locally that mujoco==3.5.0 is what mujoco-warp==3.5.0 actually needs; bump
# both together if you ever change MJLAB/mujoco-warp versions).
!pip install -q --upgrade "mujoco==3.5.0"
# mjlab==1.2.0's sim.py reaches into warp's *internal* `wp.context.runtime.driver_version` rather
# than the public `wp.get_cuda_driver_version()` API added in later mjlab releases. mjlab's own pin
# (warp-lang>=1.12.0) is a loose floor, so an unconstrained install grabs the newest warp-lang --
# but warp-lang>=1.13.0 deleted the top-level `warp/context.py` shim entirely, so `wp.context` no
# longer resolves and training crashes with `AttributeError: module 'warp' has no attribute
# 'context'` right as the sim is constructed. Pin to the last warp-lang release that still ships
# the (deprecated but functional) `warp.context` shim.
!pip install -q --upgrade "warp-lang==1.12.1"
!pip install -q huggingface_hub

import mjlab, mujoco, mujoco_warp
print("mujoco:", mujoco.__version__)
print("mujoco_warp OK:", mujoco_warp.__file__)
print("mjlab OK:", mjlab.__file__)

## 4. Hugging Face login + cross-session checkpoint sync
This server's filesystem is ephemeral, so every checkpoint is uploaded to a Hugging Face model repo as training goes (via `HF_CHECKPOINT_REPO` env var read by `PasOnPolicyRunner.save()`), and this notebook checks that repo for a checkpoint to resume from before starting.

In [ ]:
import os
from huggingface_hub import HfApi, login, whoami, create_repo, hf_hub_download

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is not set. Export it in the shell before launching Jupyter, e.g. `export HF_TOKEN=hf_...`."
    )
login(token=hf_token)

hf_username = whoami()["name"]
HF_REPO_ID = f"{hf_username}/{HF_REPO_NAME}"
create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)
print("Using Hugging Face model repo:", HF_REPO_ID)

os.environ["HF_CHECKPOINT_REPO"] = HF_REPO_ID


def latest_hf_checkpoint(stage: str):
    """Return (iteration, filename) for the highest-iteration model_*.pt under {stage}/ on HF, or None."""
    api = HfApi()
    files = api.list_repo_files(HF_REPO_ID, repo_type="model")
    candidates = []
    for f in files:
        if f.startswith(f"{stage}/model_") and f.endswith(".pt"):
            try:
                it = int(f.split("model_")[-1].split(".pt")[0])
                candidates.append((it, f))
            except ValueError:
                continue
    if not candidates:
        return None
    return max(candidates, key=lambda x: x[0])


def sync_from_hf(stage: str, local_run_name: str):
    """Download the latest {stage} checkpoint from HF into a fixed local run dir.

    Returns (resume_args, iteration): the `--agent.*` resume args to pass to train.py
    (or [] if nothing was found on HF, i.e. this stage hasn't started yet) and the
    resumed-from iteration (0 if nothing found).
    """
    found = latest_hf_checkpoint(stage)
    if found is None:
        print(f"No existing '{stage}' checkpoint on HF -- starting fresh.")
        return [], 0
    iteration, remote_path = found
    local_dir = f"{MJLAB_DIR}/logs/rsl_rl/go2_pas/{local_run_name}"
    os.makedirs(local_dir, exist_ok=True)
    local_file = hf_hub_download(HF_REPO_ID, remote_path, repo_type="model", local_dir=MJLAB_DIR)
    target = f"{local_dir}/model_{iteration}.pt"
    if local_file != target:
        import shutil
        shutil.copy(local_file, target)
    print(f"Resuming '{stage}' from iteration {iteration} ({remote_path}).")
    return [
        "--agent.resume", "True",
        "--agent.load-run", local_run_name,
        "--agent.load-checkpoint", f"model_{iteration}.pt",
    ], iteration

## 5. Stage 1 — Oracle
`anneal_prob` pinned at 1.0. Trains the terrain encoder + actor MLP via PPO, and the estimator via an auxiliary reconstruction loss only (see `docs/07-pas-implementation.md` §2).

In [ ]:
os.environ["HF_CHECKPOINT_STAGE"] = "stage1"
os.environ["MUJOCO_GL"] = "egl"

stage1_resume_args, stage1_iteration = sync_from_hf("stage1", "a100_stage1")
stage1_remaining = STAGE1_BUDGET - stage1_iteration

if stage1_remaining <= 0:
    print(f"Stage 1 already reached its {STAGE1_BUDGET}-iteration budget ({stage1_iteration} done) -- skipping.")
else:
    %cd {MJLAB_DIR}
    !python scripts/train.py Unitree-Go2-PAS-Oracle \
      --env.scene.num-envs {NUM_ENVS} \
      --agent.max-iterations {stage1_remaining} \
      --agent.save-interval {SAVE_INTERVAL} \
      --agent.logger tensorboard \
      --agent.experiment-name go2_pas \
      --gpu-ids "{GPU_IDS}" \
      {' '.join(stage1_resume_args)}

## 6. Stage 2 — Anneal
Resumes from Stage 1's final checkpoint (or continues an interrupted Stage 2 run if one already exists on HF). `anneal_prob` decays as `0.9998^iteration`, restarting fresh each time a new Stage-2 run starts (see §7 of the doc — this matters if Stage 2 itself gets interrupted and resumed).

In [ ]:
os.environ["HF_CHECKPOINT_STAGE"] = "stage2"

stage1_found = latest_hf_checkpoint("stage1")
if stage1_found is None:
    print("No 'stage1' checkpoint on HF yet -- stage 2 can't start. Run the Stage 1 cell first.")
else:
    # Stage 2's first checkpoint inherits Stage 1's absolute rsl_rl iteration count (loading
    # a checkpoint restores `current_learning_iteration` as-is, it isn't reset per stage) --
    # so STAGE2_BUDGET is counted relative to this baseline, not from 0.
    stage1_baseline = stage1_found[0]

    stage2_resume_args, stage2_iteration = sync_from_hf("stage2", "a100_stage2")
    if not stage2_resume_args:
        # First time entering Stage 2: resume from Stage 1's final checkpoint instead.
        stage2_resume_args, stage2_iteration = sync_from_hf("stage1", "a100_stage1")

    stage2_elapsed = stage2_iteration - stage1_baseline
    stage2_remaining = STAGE2_BUDGET - stage2_elapsed

    if stage2_remaining <= 0:
        print(f"Stage 2 already reached its {STAGE2_BUDGET}-iteration budget ({stage2_elapsed} done) -- skipping.")
    else:
        %cd {MJLAB_DIR}
        !python scripts/train.py Unitree-Go2-PAS-Anneal \
          --env.scene.num-envs {NUM_ENVS} \
          --agent.max-iterations {stage2_remaining} \
          --agent.save-interval {SAVE_INTERVAL} \
          --agent.logger tensorboard \
          --agent.experiment-name go2_pas \
          --agent.run-name stage2 \
          --gpu-ids "{GPU_IDS}" \
          {' '.join(stage2_resume_args)}

## 7. Next steps

- **Monitor**: `%load_ext tensorboard` then `%tensorboard --logdir {MJLAB_DIR}/logs/rsl_rl/go2_pas` in a cell, or watch the `Mean reward` / `anneal_prob` / `estimator_mse` lines printed above.
- **If the job/container gets killed**: just re-run this notebook top to bottom. Both stage cells check HF first and resume automatically — nothing is lost except whatever happened since the last `SAVE_INTERVAL` checkpoint.
- **Checkpoints** live at `hf.co/{HF_REPO_ID}` under `stage1/` and `stage2/`.
- **Visual evaluation** (needs a display / won't run headless on this server) — download a checkpoint and run locally:
  ```bash
  python scripts/play.py Unitree-Go2-PAS-Anneal --checkpoint_file <downloaded model_N.pt>
  ```
- **ONNX/deployment export** is not yet PAS-aware (see `docs/07-pas-implementation.md` §7) — the `.pt` checkpoints here are for resuming/evaluating, not yet for flashing onto a real robot.